In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a GPU program that computes the Rotary Positional Embedding (RoPE) for a batch of query vectors.
    RoPE is a method for encoding positional information in transformer models by rotating the query and key vectors using precomputed cosine and sine components.
</p>
<p>
    Mathematically, given a query vector $x$ and corresponding cosine and sine vectors, the operation is defined as:
    $$
    \text{RoPE}(x) = x \odot \cos + \text{rotate\_half}(x) \odot \sin
    $$
</p>
<p>
    Where $\odot$ denotes element-wise multiplication. The $\text{rotate\_half}(x)$ operation swaps the first and second halves of the vector and negates the first half. For a vector of dimension $d$:
    $$
    \text{rotate\_half}([x_1, \dots, x_{d/2}, x_{d/2+1}, \dots, x_d]) = [-x_{d/2+1}, \dots, -x_d, x_1, \dots, x_{d/2}]
    $$
</p>
<h2>Implementation Requirements</h2>
<ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The input tensors <code>Q</code>, <code>cos</code>, and <code>sin</code> have shape <code>(M, D)</code>, where <code>M</code> is the number of tokens and <code>D</code> is the head dimension</li>
    <li><code>D</code> (head dimension) is guaranteed to be an even number</li>
    <li>The final result must be stored in the output variable with the same shape <code>(M, D)</code></li>
</ul>
<h2>Example 1:</h2>
<pre>Input:  Q   = [[1.0, 2.0, 3.0, 4.0],
               [1.0, 1.0, 1.0, 1.0]]
        Cos = [[1.0, 1.0, 1.0, 1.0],
               [0.0, 0.0, 0.0, 0.0]]
        Sin = [[0.0, 0.0, 0.0, 0.0],
               [1.0, 1.0, 1.0, 1.0]]
Output: result = [[1.0, 2.0, 3.0, 4.0],
                  [-1.0, -1.0, 1.0, 1.0]]
        (Row 0 is identity via Cos; Row 1 is rotated via Sin)</pre>
<h2>Constraints</h2>
<ul>
    <li><code>Q</code>, <code>cos</code>, and <code>sin</code> have identical dimensions</li>
    <li><code>D</code> % 2 == 0</li>
    <li>1 ≤ <code>M</code>, <code>D</code> ≤ 10,000</li>

  <li>Performance is measured with <code>D</code> = 128, <code>M</code> = 1,048,576</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, cos, sin, output are device pointers
extern "C" void solve(float* Q, float* cos, float* sin, float* output, int M, int D) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, cos, sin, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    cos: cute.Tensor,
    sin: cute.Tensor,
    output: cute.Tensor,
    M: cute.Int32,
    D: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, cos, sin are tensors on the GPU
@jax.jit
def solve(Q: jax.Array, cos: jax.Array, sin: jax.Array, M: int, D: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# Q, cos, sin, output are device pointers
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    cos: UnsafePointer[Float32, MutExternalOrigin],
    sin: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    D: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, cos, sin, output are tensors on the GPU
def solve(
    Q: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, output: torch.Tensor, M: int, D: int
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, cos, sin, output are tensors on the GPU
def solve(
    Q: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, output: torch.Tensor, M: int, D: int
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Rotary Positional Embedding",
            atol=1e-4,
            rtol=1e-4,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        Q: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        output: torch.Tensor,
        M: int,
        D: int,
    ):
        assert Q.shape == (M, D)
        assert cos.shape == (M, D)
        assert sin.shape == (M, D)
        assert output.shape == (M, D)

        # rotate_half implementation
        # Split the last dimension into two halves
        x1 = Q[..., : D // 2]
        x2 = Q[..., D // 2 :]
        # Concatenate -x2 and x1
        rotated_Q = torch.cat((-x2, x1), dim=-1)

        # RoPE calculation
        # Output = Q * Cos + rotate_half(Q) * Sin
        result = (Q * cos) + (rotated_Q * sin)

        output.copy_(result)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "Q": (ctypes.POINTER(ctypes.c_float), "in"),
            "cos": (ctypes.POINTER(ctypes.c_float), "in"),
            "sin": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "D": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        M = 1024
        D = 64
        dtype = torch.float32

        Q = torch.randn(M, D, device="cuda", dtype=dtype)
        Cos = torch.randn(M, D, device="cuda", dtype=dtype)
        Sin = torch.randn(M, D, device="cuda", dtype=dtype)
        Output = torch.zeros(M, D, device="cuda", dtype=dtype)

        return {
            "Q": Q,
            "cos": Cos,
            "sin": Sin,
            "output": Output,
            "M": M,
            "D": D,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []
        dtype = torch.float32

        # Test 1: Small input
        M = 4
        D = 4
        tests.append(
            {
                "Q": torch.randn(M, D, device="cuda", dtype=dtype),
                "cos": torch.randn(M, D, device="cuda", dtype=dtype),
                "sin": torch.randn(M, D, device="cuda", dtype=dtype),
                "output": torch.zeros(M, D, device="cuda", dtype=dtype),
                "M": M,
                "D": D,
            }
        )

        # Test 2: Larger input
        M = 128
        D = 64
        tests.append(
            {
                "Q": torch.randn(M, D, device="cuda", dtype=dtype),
                "cos": torch.randn(M, D, device="cuda", dtype=dtype),
                "sin": torch.randn(M, D, device="cuda", dtype=dtype),
                "output": torch.zeros(M, D, device="cuda", dtype=dtype),
                "M": M,
                "D": D,
            }
        )

        # zero_matrices: outputs should remain zero when inputs are zero
        tests.append(
            {
                "Q": torch.zeros((3, 6), device="cuda", dtype=dtype),
                "cos": torch.zeros((3, 6), device="cuda", dtype=dtype),
                "sin": torch.zeros((3, 6), device="cuda", dtype=dtype),
                "output": torch.zeros(3, 6, device="cuda", dtype=dtype),
                "M": 3,
                "D": 6,
            }
        )

        # minimal_dims: smallest even D that still allows rotation
        tests.append(
            {
                "Q": torch.randn((1, 2), device="cuda", dtype=dtype),
                "cos": torch.randn((1, 2), device="cuda", dtype=dtype),
                "sin": torch.randn((1, 2), device="cuda", dtype=dtype),
                "output": torch.zeros(1, 2, device="cuda", dtype=dtype),
                "M": 1,
                "D": 2,
            }
        )

        # mixed_values: negative and positive entries
        tests.append(
            {
                "Q": torch.tensor(
                    [[-1.0, 2.0, -3.0, 4.0], [5.0, -6.0, 7.0, -8.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "cos": torch.tensor(
                    [[0.5, 0.5, 0.5, 0.5], [0.1, 0.2, 0.3, 0.4]],
                    device="cuda",
                    dtype=dtype,
                ),
                "sin": torch.tensor(
                    [[0.5, -0.5, 0.5, -0.5], [0.4, -0.3, 0.2, -0.1]],
                    device="cuda",
                    dtype=dtype,
                ),
                "output": torch.zeros(2, 4, device="cuda", dtype=dtype),
                "M": 2,
                "D": 4,
            }
        )

        # large_matrices: random uniform values for stress testing
        tests.append(
            {
                "Q": torch.empty((256, 128), device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "cos": torch.empty((256, 128), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "sin": torch.empty((256, 128), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "output": torch.zeros(256, 128, device="cuda", dtype=dtype),
                "M": 256,
                "D": 128,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        M = 1024 * 1024  # 1M tokens
        D = 128
        dtype = torch.float32
        return {
            "Q": torch.randn(M, D, device="cuda", dtype=dtype),
            "cos": torch.randn(M, D, device="cuda", dtype=dtype),
            "sin": torch.randn(M, D, device="cuda", dtype=dtype),
            "output": torch.zeros(M, D, device="cuda", dtype=dtype),
            "M": M,
            "D": D,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
